# 🔭 Projet — Analyse des Exoplanètes (NASA Exoplanet Archive)
### Analyse Exploratoire de Données · HETIC MD4

---

**Contexte**

Vous êtes data scientist au sein d'une équipe de recherche en astronomie. À partir des données consolidées du NASA Exoplanet Archive, votre mission est de caractériser la population des exoplanètes connues, d'identifier des profils planétaires distincts et d'explorer les conditions favorables à l'habitabilité.

---

**⚠️ Consignes de rendu**

Ce notebook doit être rendu **entièrement exécuté via Google Colab** (toutes les cellules doivent avoir un output visible).

**Convention à respecter tout au long du notebook :**

> Pour chaque question, votre réponse doit comporter **trois éléments** dans cet ordre :
> 1. 💻 **Une cellule de code** produisant le résultat
> 2. 📊 **Un graphique** quand cela est pertinent et possible
> 3. 💬 **Une cellule Markdown** contenant votre interprétation en quelques phrases

---

## ⚠️ Points clés à maîtriser avant de commencer

### Structure du fichier — le problème des doublons

Ce dataset n'est pas une table "une ligne = une planète".
Chaque planète peut apparaître **plusieurs fois**, une ligne par référence bibliographique : différentes équipes de chercheurs publient des mesures différentes pour la même planète au fil du temps.

La colonne `default_flag` vaut `1` pour la mesure de référence retenue par la NASA pour chaque planète.

> **Vous devez filtrer sur `default_flag == 1` dès le chargement.** Sans ce filtre, toutes vos statistiques descriptives seront faussées — une planète très étudiée pèsera artificiellement plus qu'une planète peu documentée.

### Unités physiques

Les grandeurs planétaires sont exprimées dans deux systèmes d'unités :

| Variable | Unité terrestre | Unité jovienne |
|---------|----------------|----------------|
| Rayon | `pl_rade` (rayons terrestres) | `pl_radj` (rayons de Jupiter) |
| Masse | `pl_bmasse` (masses terrestres) | `pl_bmassj` (masses de Jupiter) |

Choisissez le système le plus adapté à chaque analyse et mentionnez-le explicitement dans vos interprétations.

### Valeurs manquantes — ne pas dropna() naïvement

Certaines colonnes comme `pl_bmasse`, `pl_orbeccen` ou `pl_insol` ont des taux de valeurs manquantes très élevés (souvent > 50%). Ce n'est pas une erreur de saisie : ces paramètres sont simplement très difficiles à mesurer avec les instruments actuels.

> **Ne faites jamais `df.dropna()` sans préciser `subset`.** Une suppression naïve réduirait drastiquement le dataset et introduirait un biais de sélection majeur — les planètes les mieux caractérisées ne sont pas représentatives de l'ensemble.

### Pièges à éviter

- `pl_bmasse` et `pl_bmassj` contiennent parfois des **limites supérieures** plutôt que des mesures réelles. La colonne `pl_bmassprov` indique la provenance de la valeur — vérifiez avant d'utiliser la masse dans un modèle prédictif.
- `pl_eqt` (température d'équilibre) est une **estimation théorique**, pas une mesure directe de la température de surface.
- La distance `sy_dist` est exprimée en **parsecs** (1 pc ≈ 3,26 années-lumière) — précisez l'unité dans vos interprétations.
- Les colonnes `*err1` et `*err2` sont les **barres d'erreur** des mesures (incertitude haute et basse). Ne les utilisez pas comme variables d'analyse.

---

## Barème

| Section | Points |
|---------|--------|
| 1. Prise en main, nettoyage et qualité | 3 pts |
| 2. Analyses univariées | 2 pts |
| 3. Feature engineering | 3 pts |
| 4. Corrélations et analyses bivariées | 3 pts |
| 5. Analyse croisée | 2 pts |
| 6. Régression | 3 pts |
| 7. ACP | 2 pts |
| 8. Synthèse et recommandations | 2 pts |
| **Total** | **20 pts** |


---
## 0. Mise en place *(non noté)*

### 0.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print("✅ Imports OK")

### 0.2 Chargement et filtre de référence

Le fichier `projet_E_exoplanets.csv` doit être uploadé dans votre environnement Colab avant exécution.
Le filtre `default_flag == 1` est **obligatoire** — voir l'explication en en-tête.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Uploadez dataset_E_exoplanets.csv

df_raw = pd.read_csv('projet_E_exoplanets.csv', comment='#')
print(f"Brut : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")

df = df_raw[df_raw['default_flag'] == 1].copy()
print(f"Après filtre default_flag==1 : {df.shape[0]} lignes")
df.head()

### 0.3 Dictionnaire des variables retenues

| Colonne | Signification | Unité |
|---------|--------------|-------|
| `pl_name` | Nom de la planète | string |
| `hostname` | Nom de l'étoile hôte | string |
| `discoverymethod` | Méthode de découverte | string |
| `disc_year` | Année de découverte | entier |
| `sy_snum` | Nombre d'étoiles dans le système | entier |
| `sy_pnum` | Nombre de planètes dans le système | entier |
| `pl_orbper` | Période orbitale | jours |
| `pl_orbsmax` | Demi-grand axe orbital | UA (unités astronomiques) |
| `pl_rade` | Rayon de la planète | rayons terrestres (R⊕) |
| `pl_radj` | Rayon de la planète | rayons de Jupiter (R♃) |
| `pl_bmasse` | Masse de la planète | masses terrestres (M⊕) |
| `pl_bmassj` | Masse de la planète | masses de Jupiter (M♃) |
| `pl_orbeccen` | Excentricité orbitale | 0 (circulaire) → 1 |
| `pl_insol` | Flux d'insolation reçu | relatif à la Terre (S⊕) |
| `pl_eqt` | Température d'équilibre (estimée) | Kelvin (K) |
| `st_teff` | Température effective de l'étoile | Kelvin (K) |
| `st_rad` | Rayon de l'étoile | rayons solaires (R☉) |
| `st_mass` | Masse de l'étoile | masses solaires (M☉) |
| `st_met` | Métallicité stellaire | dex (log relatif au Soleil) |
| `st_logg` | Gravité de surface stellaire | log(cm/s²) |
| `sy_dist` | Distance du système | parsecs (1 pc ≈ 3,26 al) |


---
## Workflow à respecter

Vous avez vu en cours que tout projet d'AED suit un enchaînement logique. C'est exactement ce que vous devez reproduire ici, de façon autonome :

```
1. Comprendre les données      → structure, types, signification
2. Nettoyer                    → manquants, valeurs aberrantes, biais de sélection
3. Explorer en univarié        → distribution de chaque variable clé
4. Construire des features     → variables dérivées pertinentes pour la question
5. Explorer en bivarié         → corrélations, scatter plots
6. Analyses croisées           → groupby, heatmaps
7. Modéliser                   → régression, ACP
8. Conclure                    → insights, limites, perspectives
```

**Question scientifique centrale à garder en tête tout au long de l'analyse :**

> *Quelles caractéristiques physiques distinguent les différents types d'exoplanètes, et peut-on identifier des profils compatibles avec les critères d'habitabilité ?*

---


## 1. Prise en main, nettoyage et qualité des données *(3 pts)*

Explorez la structure du dataset filtré.
Analysez les valeurs manquantes en tenant compte des mises en garde de l'en-tête.
Identifiez et justifiez les traitements nécessaires avant toute analyse.

Combien d'exoplanètes distinctes le dataset contient-il après filtre ?
Quelles colonnes sont exploitables pour la suite, et lesquelles sont trop incomplètes ?


> *Votre interprétation ici*

## 2. Analyses univariées *(2 pts)*

Explorez la distribution des variables physiques clés : rayon, masse, période orbitale, température d'équilibre, distance.

Certaines de ces variables nécessitent-elles une transformation avant d'être analysées ?
Que révèlent ces distributions sur la population d'exoplanètes actuellement connue ?


> *Votre interprétation ici*

## 3. Feature engineering *(3 pts)*

Construisez les variables dérivées que vous jugez pertinentes pour répondre à la question scientifique centrale.

Quelques pistes (non exhaustives) : densité planétaire apparente, classification en types planétaires (rocheuse, Neptune, géante gazeuse...), indicateur de zone habitable, évolution temporelle des découvertes.

Justifiez chaque variable créée et montrez qu'elle apporte une information nouvelle par rapport aux variables brutes.


> *Votre interprétation ici*

## 4. Corrélations et analyses bivariées *(3 pts)*

Explorez les relations entre les variables physiques planétaires et stellaires.

Quelles paires de variables sont les plus corrélées ? Ces corrélations ont-elles un sens physique ?
La relation entre certaines variables est-elle mieux capturée sur une échelle logarithmique ?


> *Votre interprétation ici*

## 5. Analyse croisée *(2 pts)*

Analysez comment les caractéristiques planétaires varient selon la méthode de découverte, l'année de découverte, ou le type d'étoile hôte.

Ces différences reflètent-elles des biais instrumentaux ou de véritables différences physiques entre populations ?


> *Votre interprétation ici*

## 6. Régression *(3 pts)*

Construisez un ou plusieurs modèles de régression linéaire pertinents au regard de la question scientifique centrale.

Interprétez les coefficients, évaluez le R² et discutez les limites du modèle linéaire sur ce type de données.


> *Votre interprétation ici*

## 7. ACP *(2 pts)*

Appliquez une ACP sur les variables physiques numériques disponibles (après traitement des valeurs manquantes).

Combien de composantes sont nécessaires pour capturer l'essentiel de la variance ?
Que représentent les axes principaux en termes physiques ?
Les différents types planétaires se séparent-ils dans l'espace réduit ?


> *Votre interprétation ici*

## 8. Synthèse et recommandations *(2 pts)*

### 8.1 Insights scientifiques
Présentez vos 3 résultats les plus marquants, chacun appuyé par un chiffre ou une visualisation issue de votre analyse.

### 8.2 Limites
Quelles sont les limites de cette analyse ? Pensez aux biais de détection (les instruments actuels favorisent-ils certains types de planètes ?), à la qualité des mesures, et aux variables non disponibles.

### 8.3 Perspectives
Quelles données ou méthodes supplémentaires permettraient d'affiner la caractérisation des planètes potentiellement habitables ?


> *Votre synthèse ici*